# Deep Learning for NLP

**Deep Learning** is a type of Machine Learning that uses **Neural Networks with multiple layers** to learn patterns from data.

Deep Learning can be used in NLP tasks such as:
- Text Classification
- Sentiment Analysis
- Question Answering

In this lab, we will build a **deep learning model** to classify Yelp restaurant reviews as:

- **0 = Negative**
- **1 = Positive**

<img src="https://drive.google.com/uc?export=view&id=1rUaLO1wEOcUZ10Kwd0nUvS3jQzrqoOcI" width="550">
<img src="https://drive.google.com/uc?export=view&id=1XKPTKwxmIpP5cZIkDajlai6aI9jUX-C8" width="400">

## 1. Load the Yelp Dataset

Each row contains:

- a restaurant review
- its sentiment label


In [ ]:
import pandas as pd

df = pd.read_csv("07-yelp-dataset.txt", sep="\t", names=["review", "label"])

print("Number of reviews:", len(df))
df.head()

Number of reviews: 1000


,review,label
0,Wow... Loved this place.,1
1,Crust is not good.,0
2,Not tasty and the texture was just nasty.,0
3,Stopped by during the late May bank holiday of...,1
4,The selection on the menu was great and so wer...,1


In [ ]:
print(df["label"].value_counts())

label
1    500
0    500
Name: count, dtype: int64


## 2. Preprocessing the Dataset

In [ ]:
import re

def preprocess_text(text):

    # 1. Convert to lowercase
    text = text.lower()

    # 2. Remove punctuation and special characters
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # 3. Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

    df["review"] = df["review"].apply(preprocess_text)
    df.head()

## 3. Split the Data

We divide the dataset into:

- **80% training data** — used to train the model
- **20% testing data** — used to evaluate the model

`stratify=y` keeps the proportion of positive and negative reviews similar in both sets.

In [ ]:
from sklearn.model_selection import train_test_split

X = df["review"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

print("Training reviews:", len(X_train))
print("Testing reviews:", len(X_test))

Training reviews: 800
Testing reviews: 200


## 4. Convert Text to TF-IDF Vectors

A neural network cannot directly understand text.

**TF-IDF converts each review into a vector of numbers.**


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=1000, lowercase=True)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Testing TF-IDF shape:", X_test_tfidf.shape)

Training TF-IDF shape: (800, 1000)
Testing TF-IDF shape: (200, 1000)


## 5. Convert the Data to PyTorch Tensors

PyTorch neural networks work with **tensors**.

In [ ]:
import torch
import torch.nn as nn

# Convert TF-IDF data to PyTorch tensors
X_train_tensor = torch.FloatTensor(X_train_tfidf.toarray())
X_test_tensor = torch.FloatTensor(X_test_tfidf.toarray())

y_train_tensor = torch.FloatTensor(y_train.values).reshape(-1, 1)

## 6. Build a Simple Neural Network

Our network has:

1. **Input layer** — receives the TF-IDF features
2. **Hidden layer** — learns useful patterns
3. **ReLU activation** — adds non-linearity
4. **Output layer** — produces one value for binary classification

We do **not** put `Sigmoid` inside the model because `BCEWithLogitsLoss` applies it internally in a numerically stable way.

In [ ]:
input_size = X_train_tensor.shape[1]
# torch.size([800, 1000])

model = nn.Sequential(
    nn.Linear(input_size, 64),   # Input layer
    nn.ReLU(),

    nn.Linear(64, 32),           # Hidden layer
    nn.ReLU(),

    nn.Linear(32, 1)             # Output layer
)

print(model)

Sequential(
  (0): Linear(in_features=1000, out_features=64, bias=True)
  (1): ReLU()
  (2): Linear(in_features=64, out_features=32, bias=True)
  (3): ReLU()
  (4): Linear(in_features=32, out_features=1, bias=True)
)


## 7. Choose the Loss Function and Optimizer

- **BCEWithLogitsLoss** is used for binary classification.
To measure the model’s mistakes.
- **Adam** is responsible for changing the model's weights so that the loss becomes smaller.

In [ ]:
loss_function = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

## 8. Train the Neural Network

One **epoch** means the model has seen the full training dataset once.

For every epoch:

1. Make predictions
2. Calculate the loss
3. Calculate gradients with backpropagation
4. Update the weights

In [ ]:
epochs = 10

for epoch in range(epochs):
    model.train()

    # 1. Forward pass
    outputs = model(X_train_tensor)

    # 2. Calculate loss
    loss = loss_function(outputs, y_train_tensor)

    # 3. Backpropagation
    loss.backward()

    # 4. Update model weights
    optimizer.step()

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch + 1}/{epochs} - Loss: {loss.item():.4f}")

Epoch 5/10 - Loss: 0.6517
Epoch 10/10 - Loss: 0.4769


## 9. Evaluate the Model

The model outputs raw scores called **logits**.

We apply `sigmoid` to convert them to probabilities between 0 and 1.

- Probability **≥ 0.5 → Positive (1)**
- Probability **< 0.5 → Negative (0)**

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

model.eval()

with torch.no_grad():
    test = model(X_test_tensor)
    test_probabilities = torch.sigmoid(test)
    predictions = (test_probabilities >= 0.5).int().numpy().flatten()

accuracy = accuracy_score(y_test, predictions)

print(f"Test Accuracy: {accuracy:.3f}")

Test Accuracy: 0.755


## 10. Classification Report and Confusion Matrix

The classification report shows:

- **Precision**
- **Recall**
- **F1-score**

The confusion matrix shows how many reviews were classified correctly and incorrectly.

In [ ]:
print("Classification Report:")
print(classification_report(y_test, predictions, target_names=["Negative", "Positive"]))

print("Confusion Matrix:")
print(confusion_matrix(y_test, predictions))

Classification Report:
              precision    recall  f1-score   support

    Negative       0.68      0.94      0.79        96
    Positive       0.91      0.59      0.71       104

    accuracy                           0.76       200
   macro avg       0.79      0.76      0.75       200
weighted avg       0.80      0.76      0.75       200

Confusion Matrix:
[[90  6]
 [43 61]]


## Task 1 – Represent Reviews Using Word2Vec

In the previous part of the lab, TF-IDF was used to convert each review into numerical features.
Now, use Word2Vec instead.

Requirements:
Tokenize the training and testing reviews.
Train a Word2Vec model using the training reviews.
Use:
vector_size = 200
window = 5
min_count = 1
sg = 1

Convert each review into a single vector.
Convert the resulting vectors into PyTorch tensors.


In [4]:
import numpy as np
import torch
from gensim.models import Word2Vec

# Apply the preprocessing function defined above to the train / test reviews
X_train_clean = X_train.apply(preprocess_text)
X_test_clean = X_test.apply(preprocess_text)

# 1. Tokenize the training and testing reviews
train_tokens = [review.split() for review in X_train_clean]
test_tokens = [review.split() for review in X_test_clean]

# 2. Train Word2Vec on the training reviews only
# The dataset is very small (~800 reviews), so we train for 100 epochs instead of the default 5
w2v_model = Word2Vec(train_tokens, vector_size=200, window=5, min_count=1, sg=1, seed=42, workers=1, epochs=100)
print("Vocabulary size:", len(w2v_model.wv))

# 3. Convert each review into one vector = average of its word vectors
def review_to_vector(tokens, model):
    vectors = [model.wv[word] for word in tokens if word in model.wv]
    if len(vectors) == 0:                       # review with no known words
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)

X_train_w2v = np.array([review_to_vector(t, w2v_model) for t in train_tokens])
X_test_w2v = np.array([review_to_vector(t, w2v_model) for t in test_tokens])

# Standardize the features (averaged word vectors have very small values, which slows down training)
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_w2v = scaler.fit_transform(X_train_w2v)   # fit only on training data
X_test_w2v = scaler.transform(X_test_w2v)

# 4. Convert to PyTorch tensors
X_train_w2v_tensor = torch.FloatTensor(X_train_w2v)
X_test_w2v_tensor = torch.FloatTensor(X_test_w2v)
y_train_w2v_tensor = torch.FloatTensor(y_train.values).reshape(-1, 1)

print("Training tensor shape:", X_train_w2v_tensor.shape)
print("Testing tensor shape:", X_test_w2v_tensor.shape)

Vocabulary size: 1801
Training tensor shape: torch.Size([800, 200])
Testing tensor shape: torch.Size([200, 200])


## Task 2 – Build a Different Neural Network

Use the Word2Vec vectors from Task 1 as the input to a new neural network.
Your network must contain:
An input layer
Three hidden layers
ReLU activation functions
One output neuron
Use the following architecture:

Input
   
   ↓

128 neurons
   
   ↓

64 neurons
   
   ↓

32 neurons
   
   ↓

1 output neuron

Train the model using:
, BCEWithLogitsLoss
, Adam optimizer
, Learning rate = 0.001
, 15 epochs.
Then calculate the test accuracy.

In [5]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score

torch.manual_seed(42)

# Input -> 128 -> 64 -> 32 -> 1
w2v_nn = nn.Sequential(
    nn.Linear(X_train_w2v_tensor.shape[1], 128),
    nn.ReLU(),

    nn.Linear(128, 64),
    nn.ReLU(),

    nn.Linear(64, 32),
    nn.ReLU(),

    nn.Linear(32, 1)
)
print(w2v_nn)

loss_function = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(w2v_nn.parameters(), lr=0.001)

# Mini-batches of 32 reviews: the weights are updated several times per epoch
train_loader = DataLoader(TensorDataset(X_train_w2v_tensor, y_train_w2v_tensor), batch_size=32, shuffle=True)

epochs = 15
for epoch in range(epochs):
    w2v_nn.train()
    total_loss = 0
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()                       # reset gradients from the previous step
        outputs = w2v_nn(batch_X)                   # forward pass
        loss = loss_function(outputs, batch_y)      # calculate loss
        loss.backward()                             # backpropagation
        optimizer.step()                            # update weights
        total_loss += loss.item() * len(batch_X)
    print(f"Epoch {epoch + 1}/{epochs} - Loss: {total_loss / len(X_train_w2v_tensor):.4f}")

Sequential(
  (0): Linear(in_features=200, out_features=128, bias=True)
  (1): ReLU()
  (2): Linear(in_features=128, out_features=64, bias=True)
  (3): ReLU()
  (4): Linear(in_features=64, out_features=32, bias=True)
  (5): ReLU()
  (6): Linear(in_features=32, out_features=1, bias=True)
)


Epoch 1/15 - Loss: 0.6553
Epoch 2/15 - Loss: 0.5184


Epoch 3/15 - Loss: 0.3988
Epoch 4/15 - Loss: 0.3126
Epoch 5/15 - Loss: 0.2249
Epoch 6/15 - Loss: 0.1568
Epoch 7/15 - Loss: 0.1041


Epoch 8/15 - Loss: 0.0743
Epoch 9/15 - Loss: 0.0482
Epoch 10/15 - Loss: 0.0405
Epoch 11/15 - Loss: 0.0234
Epoch 12/15 - Loss: 0.0180


Epoch 13/15 - Loss: 0.0172
Epoch 14/15 - Loss: 0.0212
Epoch 15/15 - Loss: 0.0412


In [6]:
# Test accuracy
w2v_nn.eval()
with torch.no_grad():
    test_probabilities = torch.sigmoid(w2v_nn(X_test_w2v_tensor))
    w2v_predictions = (test_probabilities >= 0.5).int().numpy().flatten()

print(f"Word2Vec + NN Test Accuracy: {accuracy_score(y_test, w2v_predictions):.3f}")

Word2Vec + NN Test Accuracy: 0.730


**Note:** Word2Vec is trained here on only ~800 short reviews, so the word vectors are weak (Word2Vec usually needs millions of words to learn good meanings). With the default 5 training epochs and unscaled features, the network could not learn at all (loss stayed at 0.693, accuracy ≈ 50%), which is why Word2Vec is trained for 100 epochs and the features are standardized. Averaging the word vectors also loses word order and dilutes strong sentiment words like *"not"* or *"terrible"*, so this model still scores lower than the TF-IDF model. Using pre-trained embeddings (e.g. Google News Word2Vec or GloVe) would usually improve the result.